# Full pipeline rerun: DNN ensemble, NGBoost and error analysis

Based on `02_DNN_RGC_pretraining_and_finetuning.ipynb` and `03_NGBoost_real_vs_RGC_augmented.ipynb`. Produces the verified predictions and the error-by-temperature-range analysis prepared for the revision.

In [ ]:
!pip install ngboost -q

In [ ]:
# 0) Imports & config
import os, zipfile, datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from google.colab import files

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from ngboost import NGBRegressor
from ngboost.distns import Normal

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

FEATURES = [
    'pH','EC (microS/cm)','K (mg/l)','Na (mg/l)',
    'Boron (mg/l)','SiO2 (mg/l)','Cl (mg/l)'
]
TARGET = 'Reservoir temperature (°C)'

DNN_CONFIG = {
    "n_ensembles": 10,
    "pre_epochs": 120,
    "fine_epochs": 200,
    "batch_size": 16,
    "lr_pre": 1e-3,
    "lr_fine": 3e-4,
}

OUT_DIR = "pipeline_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Manuscript Table 5c reference
MANUSCRIPT = {
    "DNN_RGC":      {"RMSE": 4.12, "MAE": 3.2},
    "NGBoost_Real": {"RMSE": 5.23, "MAE": 4.1},
    "NGBoost_RGC":  {"RMSE": 4.26, "MAE": 3.31},
}

def rmse(y_true, y_hat):
    return np.sqrt(mean_squared_error(y_true, y_hat))

print("Setup complete.")

In [ ]:
# 1) Upload files
print("Upload 'training_dataset.csv':")
files.upload()
print("\nUpload 'testing_dataset.csv':")
files.upload()
print("\nUpload the synthetic data file (synthetic_best.csv or synthetic_rgc_train_only.csv):")
uploaded_synth = files.upload()

# Auto-detect synthetic filename
synth_filename = list(uploaded_synth.keys())[0]
print(f"\nSynthetic file: {synth_filename}")

real_train = pd.read_csv("training_dataset.csv")
real_test  = pd.read_csv("testing_dataset.csv")
synthetic  = pd.read_csv(synth_filename)
synthetic  = synthetic[FEATURES + [TARGET]]

print(f"\nTraining: {len(real_train)}, Test: {len(real_test)}, Synthetic: {len(synthetic)}")

## STAGE 1: DNN Ensemble (from DNNvTemp.ipynb)

In [ ]:
# 2) Feature/target arrays + scaling
X_train_real = real_train[FEATURES].values
y_train_real = real_train[TARGET].values
X_test_real  = real_test[FEATURES].values
y_test_real  = real_test[TARGET].values
X_synth      = synthetic[FEATURES].values
y_synth      = synthetic[TARGET].values

scaler = StandardScaler()
X_train_real_scaled = scaler.fit_transform(X_train_real)
X_test_real_scaled  = scaler.transform(X_test_real)
X_synth_scaled      = scaler.transform(X_synth)

# 3) DNN model builder + training
def build_dnn(input_dim, lr):
    reg = regularizers.l2(1e-4)
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation='relu', kernel_regularizer=reg),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu', kernel_regularizer=reg),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation='relu', kernel_regularizer=reg),
        BatchNormalization(),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

def train_with_pretrain(seed_offset=0):
    np.random.seed(GLOBAL_SEED + seed_offset)
    tf.random.set_seed(GLOBAL_SEED + seed_offset)

    model = build_dnn(X_train_real_scaled.shape[1], DNN_CONFIG["lr_pre"])
    model.fit(X_synth_scaled, y_synth,
              epochs=DNN_CONFIG["pre_epochs"], batch_size=DNN_CONFIG["batch_size"],
              verbose=0, callbacks=[EarlyStopping(monitor='loss', patience=15, restore_best_weights=True, verbose=0)])

    model.compile(optimizer=Adam(learning_rate=DNN_CONFIG["lr_fine"]), loss='mse', metrics=['mae'])
    model.fit(X_train_real_scaled, y_train_real,
              validation_data=(X_test_real_scaled, y_test_real),
              epochs=DNN_CONFIG["fine_epochs"], batch_size=DNN_CONFIG["batch_size"],
              verbose=0, callbacks=[
                  EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=0),
                  ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=0, min_lr=1e-5)])

    return model, model.predict(X_test_real_scaled, verbose=0).ravel()

# Train ensemble
print("Training DNN ensemble (RGC-pretrained)...")
all_test_preds = []
for k in range(DNN_CONFIG["n_ensembles"]):
    print(f"  Member {k+1}/{DNN_CONFIG['n_ensembles']}")
    _, pred_k = train_with_pretrain(seed_offset=k * 100)
    all_test_preds.append(pred_k)

all_test_preds = np.stack(all_test_preds)
y_pred_dnn_rgc = all_test_preds.mean(axis=0)
y_pred_dnn_std = all_test_preds.std(axis=0)

print(f"\nDNN Ensemble (RGC): RMSE={rmse(y_test_real, y_pred_dnn_rgc):.2f}, "
      f"MAE={mean_absolute_error(y_test_real, y_pred_dnn_rgc):.2f}, "
      f"R²={r2_score(y_test_real, y_pred_dnn_rgc):.4f}")

## STAGE 2: NGBoost (from 03_NGBoost notebook)

In [ ]:
# 4) NGBoost feature engineering + training
def build_feature_matrix(df_in):
    df = df_in.copy()
    X = df[FEATURES].copy()
    for col in ['EC (microS/cm)', 'Na (mg/l)', 'Cl (mg/l)', 'SiO2 (mg/l)']:
        X[f'log_{col}'] = np.log1p(np.clip(df[col].values, a_min=0, a_max=None))
    eps = 1e-6
    X['Na_plus_K_over_Cl'] = (df['Na (mg/l)'] + df['K (mg/l)']) / (df['Cl (mg/l)'] + eps)
    X['Na_over_Cl']   = df['Na (mg/l)'] / (df['Cl (mg/l)'] + eps)
    X['B_over_Cl']    = df['Boron (mg/l)'] / (df['Cl (mg/l)'] + eps)
    X['B_over_EC']    = df['Boron (mg/l)'] / (df['EC (microS/cm)'] + eps)
    X['SiO2_over_EC'] = df['SiO2 (mg/l)'] / (df['EC (microS/cm)'] + eps)
    return X

X_tr_ngb_real = build_feature_matrix(real_train)
y_tr_ngb_real = real_train[TARGET].values
X_te_ngb      = build_feature_matrix(real_test)
y_te_ngb      = real_test[TARGET].values

X_tr_ngb_rgc = pd.concat([X_tr_ngb_real, build_feature_matrix(synthetic)], ignore_index=True)
y_tr_ngb_rgc = np.concatenate([y_tr_ngb_real, synthetic[TARGET].values])

def fit_ngb(X_train, y_train, random_state=0):
    ngb = NGBRegressor(Dist=Normal, n_estimators=500, learning_rate=0.03,
                       minibatch_frac=1.0, col_sample=1.0, verbose=False,
                       random_state=random_state)
    ngb.fit(X_train, y_train)
    return ngb

print("Training NGBoost (real-only)...")
ngb_real = fit_ngb(X_tr_ngb_real, y_tr_ngb_real, random_state=GLOBAL_SEED)
y_pred_ngb_real = ngb_real.pred_dist(X_te_ngb).loc

print("Training NGBoost (RGC-augmented)...")
ngb_rgc = fit_ngb(X_tr_ngb_rgc, y_tr_ngb_rgc, random_state=GLOBAL_SEED)
y_pred_ngb_rgc = ngb_rgc.pred_dist(X_te_ngb).loc

print(f"\nNGBoost (Real):  RMSE={rmse(y_te_ngb, y_pred_ngb_real):.2f}, "
      f"MAE={mean_absolute_error(y_te_ngb, y_pred_ngb_real):.2f}, "
      f"R²={r2_score(y_te_ngb, y_pred_ngb_real):.4f}")
print(f"NGBoost (RGC):   RMSE={rmse(y_te_ngb, y_pred_ngb_rgc):.2f}, "
      f"MAE={mean_absolute_error(y_te_ngb, y_pred_ngb_rgc):.2f}, "
      f"R²={r2_score(y_te_ngb, y_pred_ngb_rgc):.4f}")

## STAGE 3: Consistency Check

In [ ]:
# 5) Consistency check vs manuscript
computed = {
    "DNN_RGC":      {"RMSE": rmse(y_test_real, y_pred_dnn_rgc),
                      "MAE": mean_absolute_error(y_test_real, y_pred_dnn_rgc),
                      "R2": r2_score(y_test_real, y_pred_dnn_rgc)},
    "NGBoost_Real": {"RMSE": rmse(y_te_ngb, y_pred_ngb_real),
                      "MAE": mean_absolute_error(y_te_ngb, y_pred_ngb_real),
                      "R2": r2_score(y_te_ngb, y_pred_ngb_real)},
    "NGBoost_RGC":  {"RMSE": rmse(y_te_ngb, y_pred_ngb_rgc),
                      "MAE": mean_absolute_error(y_te_ngb, y_pred_ngb_rgc),
                      "R2": r2_score(y_te_ngb, y_pred_ngb_rgc)},
}

print("=" * 90)
print("CONSISTENCY CHECK vs MANUSCRIPT (Table 5c)")
print("=" * 90)
print(f"{'Model':<20} {'Metric':<6} {'Manuscript':>10} {'This Run':>10} {'Diff':>8} {'Status':>10}")
print("-" * 70)

all_ok = True
for key in MANUSCRIPT:
    for metric in ['RMSE', 'MAE']:
        ms = MANUSCRIPT[key][metric]
        comp = computed[key][metric]
        diff = abs(comp - ms)
        tol = 0.5 if 'DNN' in key else 0.15
        ok = diff <= tol
        if not ok: all_ok = False
        print(f"{key:<20} {metric:<6} {ms:>10.2f} {comp:>10.2f} {diff:>8.2f} {'' if ok else '':>10}")

print("=" * 90)
print("\nR² values (Reviewer #1):")
for key in computed:
    print(f"  {key:<20} R² = {computed[key]['R2']:.4f}")

if all_ok:
    print("\n ALL OK, predictions are consistent with manuscript.")
else:
    print("\n MISMATCHES found. Check if you uploaded the ORIGINAL synthetic_best.csv.")
    print("  DNN tolerans: ±0.5°C (stochastic). NGBoost tolerans: ±0.15°C.")
    print("  If close, this may be acceptable GPU/CPU floating-point variance.")

## STAGE 4: Save Predictions

In [ ]:
# 6) Save prediction CSVs
pd.DataFrame({
    "True_T": y_test_real,
    "Pred_T_ensemble": y_pred_dnn_rgc,
    "Pred_T_std": y_pred_dnn_std,
}).to_csv(os.path.join(OUT_DIR, "test_predictions_rgc_ensemble.csv"), index=False)

pd.DataFrame({
    "True_T": y_te_ngb,
    "Pred_T_real_only": y_pred_ngb_real,
}).to_csv(os.path.join(OUT_DIR, "preds_ngb_real_only.csv"), index=False)

pd.DataFrame({
    "True_T": y_te_ngb,
    "Pred_T_rgc_aug": y_pred_ngb_rgc,
}).to_csv(os.path.join(OUT_DIR, "preds_ngb_rgc_augmented.csv"), index=False)

print("Prediction CSVs saved.")

## STAGE 5: Error-by-Range Analysis (Reviewer #2)

In [ ]:
# 7) Error by temperature range
reviewer_ranges = [
    ("<70°C",     -np.inf, 70),
    ("70–120°C",  70,      120),
    ("120–180°C", 120,     180),
    (">180°C",    180,     np.inf),
]

models_dict = {
    "DNN Ensemble (RGC)": y_pred_dnn_rgc,
    "NGBoost (Real-only)": y_pred_ngb_real,
    "NGBoost (RGC-aug.)": y_pred_ngb_rgc,
}

rows = []
for label, lo, hi in reviewer_ranges:
    mask = (y_test_real >= lo) & (y_test_real < hi)
    if mask.sum() == 0: continue
    for mname, preds in models_dict.items():
        yt, yp = y_test_real[mask], preds[mask]
        rows.append({
            "Range": label, "n": int(mask.sum()), "Model": mname,
            "RMSE": round(rmse(yt, yp), 2),
            "MAE": round(mean_absolute_error(yt, yp), 2),
            "Bias": round(float(np.mean(yp - yt)), 2),
            "Max|Error|": round(float(np.max(np.abs(yp - yt))), 2),
            "R²": round(r2_score(yt, yp), 4) if mask.sum() >= 2 else None,
        })

for mname, preds in models_dict.items():
    rows.append({
        "Range": "Overall", "n": len(y_test_real), "Model": mname,
        "RMSE": round(rmse(y_test_real, preds), 2),
        "MAE": round(mean_absolute_error(y_test_real, preds), 2),
        "Bias": round(float(np.mean(preds - y_test_real)), 2),
        "Max|Error|": round(float(np.max(np.abs(preds - y_test_real))), 2),
        "R²": round(r2_score(y_test_real, preds), 4),
    })

error_df = pd.DataFrame(rows)
error_df.to_csv(os.path.join(OUT_DIR, "error_by_temp_range.csv"), index=False)
print(error_df.to_string(index=False))

In [ ]:
# 8) Per-sample error table
detail = pd.DataFrame({
    "Sample": real_test["Sample name"].values if "Sample name" in real_test.columns else range(len(y_test_real)),
    "Location": real_test["Location"].values if "Location" in real_test.columns else "",
    "True_T": y_test_real,
    "DNN_RGC_Pred": np.round(y_pred_dnn_rgc, 1),
    "DNN_RGC_Err": np.round(y_pred_dnn_rgc - y_test_real, 1),
    "NGB_Real_Pred": np.round(y_pred_ngb_real, 1),
    "NGB_Real_Err": np.round(y_pred_ngb_real - y_test_real, 1),
    "NGB_RGC_Pred": np.round(y_pred_ngb_rgc, 1),
    "NGB_RGC_Err": np.round(y_pred_ngb_rgc - y_test_real, 1),
}).sort_values("True_T")

detail.to_csv(os.path.join(OUT_DIR, "per_sample_errors.csv"), index=False)
print(detail.to_string(index=False))

In [ ]:
# 9) Figure B4: Error analysis visualization
plt.rcParams.update({'font.family': 'serif', 'font.size': 11, 'figure.dpi': 200})

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)
colors_map = {'<70°C': '#3288BD', '70–120°C': '#66C2A5', '120–180°C': '#FEE08B', '>180°C': '#D53E4F'}
lims = [y_test_real.min() - 10, y_test_real.max() + 10]

# (a) DNN RGC scatter
ax1 = fig.add_subplot(gs[0, 0])
for label, lo, hi in reviewer_ranges:
    mask = (y_test_real >= lo) & (y_test_real < hi)
    if mask.sum() > 0:
        ax1.scatter(y_test_real[mask], y_pred_dnn_rgc[mask], c=colors_map[label],
                   s=60, label=label, edgecolors='black', linewidth=0.5, zorder=3)
ax1.plot(lims, lims, 'k--', lw=1, alpha=0.5)
ax1.set_xlabel('True T (°C)')
ax1.set_ylabel('Predicted T (°C)')
ax1.set_title('(a) DNN Ensemble (RGC)')
ax1.legend(fontsize=8)

# (b) NGBoost RGC scatter
ax2 = fig.add_subplot(gs[0, 1])
for label, lo, hi in reviewer_ranges:
    mask = (y_test_real >= lo) & (y_test_real < hi)
    if mask.sum() > 0:
        ax2.scatter(y_test_real[mask], y_pred_ngb_rgc[mask], c=colors_map[label],
                   s=60, label=label, edgecolors='black', linewidth=0.5, zorder=3)
ax2.plot(lims, lims, 'k--', lw=1, alpha=0.5)
ax2.set_xlabel('True T (°C)')
ax2.set_ylabel('Predicted T (°C)')
ax2.set_title('(b) NGBoost (RGC-aug.)')
ax2.legend(fontsize=8)

# (c) Residuals
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(y_test_real, y_pred_dnn_rgc - y_test_real, s=50, alpha=0.7, label='DNN (RGC)', marker='o', color='#2166AC')
ax3.scatter(y_test_real, y_pred_ngb_rgc - y_test_real, s=50, alpha=0.7, label='NGBoost (RGC)', marker='s', color='#B2182B')
ax3.scatter(y_test_real, y_pred_ngb_real - y_test_real, s=50, alpha=0.7, label='NGBoost (Real)', marker='^', color='#66C2A5')
ax3.axhline(0, color='black', lw=1, ls='--', alpha=0.5)
ax3.axvspan(y_test_real.min()-5, 70, alpha=0.08, color='blue')
ax3.axvspan(180, y_test_real.max()+5, alpha=0.08, color='red')
ax3.set_xlabel('True T (°C)')
ax3.set_ylabel('Residual (°C)')
ax3.set_title('(c) Residuals')
ax3.legend(fontsize=8)

# (d) MAE by range
ax4 = fig.add_subplot(gs[1, 0:2])
active_ranges, mae_dnn, mae_ngb_r, mae_ngb_a = [], [], [], []
for label, lo, hi in reviewer_ranges:
    mask = (y_test_real >= lo) & (y_test_real < hi)
    if mask.sum() == 0: continue
    active_ranges.append(f"{label}\n(n={mask.sum()})")
    mae_dnn.append(mean_absolute_error(y_test_real[mask], y_pred_dnn_rgc[mask]))
    mae_ngb_r.append(mean_absolute_error(y_test_real[mask], y_pred_ngb_real[mask]))
    mae_ngb_a.append(mean_absolute_error(y_test_real[mask], y_pred_ngb_rgc[mask]))

x_pos = np.arange(len(active_ranges))
w = 0.25
for i, (vals, col, lbl) in enumerate([
    (mae_dnn, '#2166AC', 'DNN (RGC)'),
    (mae_ngb_r, '#66C2A5', 'NGBoost (Real)'),
    (mae_ngb_a, '#B2182B', 'NGBoost (RGC)')]):
    bars = ax4.bar(x_pos + (i-1)*w, vals, w, color=col, alpha=0.8, label=lbl)
    for bar in bars:
        ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(active_ranges)
ax4.set_ylabel('MAE (°C)')
ax4.set_title('(d) MAE by temperature range')
ax4.legend(fontsize=9)

# (e) Per-sample error
ax5 = fig.add_subplot(gs[1, 2])
sort_idx = np.argsort(y_test_real)
x_s = np.arange(len(y_test_real))
ax5.bar(x_s-0.2, np.abs(y_pred_dnn_rgc[sort_idx]-y_test_real[sort_idx]), 0.4, color='#2166AC', alpha=0.7, label='DNN (RGC)')
ax5.bar(x_s+0.2, np.abs(y_pred_ngb_rgc[sort_idx]-y_test_real[sort_idx]), 0.4, color='#B2182B', alpha=0.7, label='NGBoost (RGC)')
ax5.set_xticks(x_s)
ax5.set_xticklabels([f'{t:.0f}' for t in y_test_real[sort_idx]], rotation=60, fontsize=7)
ax5.set_xlabel('True T (°C)')
ax5.set_ylabel('|Error| (°C)')
ax5.set_title('(e) Per-sample |error|')
ax5.legend(fontsize=8)

fig.suptitle('Figure B4: Prediction error analysis by temperature range', fontsize=14, fontweight='bold', y=1.01)
fig_path = os.path.join(OUT_DIR, 'fig_B4_error_by_range.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

In [ ]:
# 10) Winsorization check on extreme-T test samples
na_thresh = real_train['Na (mg/l)'].quantile(0.99)
cl_thresh = real_train['Cl (mg/l)'].quantile(0.99)

print(f"Winsorization thresholds: Na+ = {na_thresh:.1f}, Cl- = {cl_thresh:.1f}")
print(f"\nExtreme-T test samples (<70°C or >180°C):")
print(f"{'Sample':<15} {'T':>5} {'Na':>7} {'Na?':>6} {'Cl':>7} {'Cl?':>6}")
for i in range(len(real_test)):
    t = real_test[TARGET].iloc[i]
    if t < 70 or t > 180:
        na = real_test['Na (mg/l)'].iloc[i]
        cl = real_test['Cl (mg/l)'].iloc[i]
        nm = real_test['Sample name'].iloc[i] if 'Sample name' in real_test.columns else i
        print(f"{str(nm):<15} {t:>5.0f} {na:>7.0f} {'CLIP' if na > na_thresh else 'ok':>6} {cl:>7.0f} {'CLIP' if cl > cl_thresh else 'ok':>6}")

In [ ]:
# 11) Download all outputs as ZIP
ts = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
zip_name = f'pipeline_results_{ts}.zip'
zip_path = os.path.join(OUT_DIR, '..', zip_name)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(OUT_DIR):
        zf.write(os.path.join(OUT_DIR, fn), arcname=fn)

print(f"\n All outputs zipped: {zip_name}")
print("\nOutputs:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f}")

files.download(zip_path)